In [ ]:
# ── Data Handling ──
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ── Visualization ──
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# ── Preprocessing ──
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.model_selection import train_test_split

# ── Classification Models ──
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# ── Regression Models ──
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

# ── Metrics ──
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, 
    classification_report, confusion_matrix,
    mean_squared_error, mean_absolute_error, r2_score
)

import joblib
import mlflow
import mlflow.sklearn

print("✅ All libraries imported successfully.")

In [ ]:
House_Price_df = pd.read_csv('/Users/ujjwalraj/Desktop/Project Import Material 1/india_housing_prices.csv')

In [ ]:
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

House_Price_df.head(5)

In [ ]:
House_Price_df.dtypes

In [ ]:
House_Price_df.describe().round(2)

In [ ]:
# Dropping ID Column 
House_Price_df.drop(columns = ['ID'], inplace = True)

In [ ]:
# Checking Missing Values 
missing = House_Price_df.isnull().sum()

if missing.sum() == 0:
    print('No Missing Values')
else:
    print('Missing Values')

In [ ]:
# Checking Duplicates 
duplicate = House_Price_df.duplicated().sum()

if duplicate > 0:
    House_Price_df.drop_duplicates(inplace = True)
    print(f'Removed{duplicate} duplicate rows. New Shape: {House_Price_df.shape}')
else:
    print('No Duplicates')

In [ ]:
# Recalculating the Price_per_sqft
House_Price_df['Price_per_SqFt'] = House_Price_df['Price_in_Lakhs'] / House_Price_df['Size_in_SqFt']
print(House_Price_df['Price_per_SqFt'].describe().round(4))

# Feature Engineering 

In [ ]:
amenity_list = ['Gym', 'Pool', 'Garden', 'Playground', 'Clubhouse']

House_Price_df['Amenities_Count'] = House_Price_df['Amenities'].str.split(', ').str.len()

for amenity in amenity_list:
    House_Price_df[f'Has_{amenity}'] = House_Price_df['Amenities'].str.contains(amenity, case=False).astype(int)
    

In [ ]:
# Size per BHK 
House_Price_df['Size_per_BHK'] = House_Price_df['Size_in_SqFt'] / House_Price_df['BHK']

# Floor Ratio how high person is living 
House_Price_df['Floor_Ratio'] = House_Price_df['Floor_No'] / House_Price_df['Total_Floors'].clip(lower=1)

# Is_high Floor Top(30%)
House_Price_df['Is_High_Floor'] = (House_Price_df['Floor_Ratio'] > 0.7).astype(int)

# Property Age Bucket 
House_Price_df['Is_New_Property'] = (House_Price_df['Age_of_Property'] <= 5).astype(int)
House_Price_df['Is_Old_Property'] = (House_Price_df['Age_of_Property'] > 25).astype(int)

# School Density Score (normalized)
House_Price_df['School_Density_Score'] = House_Price_df['Nearby_Schools'] / 10.0

# Hospital Density Score (normalized)
House_Price_df['Hospital_Density_Score'] = House_Price_df['Nearby_Hospitals'] / 10.0

# Infrastructure score 
transport_map = {'High': 3, 'Medium': 2, 'Low': 1}
House_Price_df['Transport_Score'] = House_Price_df['Public_Transport_Accessibility'].map(transport_map)

House_Price_df['Infrastructure_Score'] = (
    House_Price_df['School_Density_Score'] +
    House_Price_df['Hospital_Density_Score'] +
    House_Price_df['Transport_Score'] / 3.0
) / 3.0

# Binary encodings
House_Price_df['Has_Parking'] = (House_Price_df['Parking_Space'] == 'Yes').astype(int)
House_Price_df['Has_Security'] = (House_Price_df['Security'] == 'Yes').astype(int)
House_Price_df['Is_Ready'] = (House_Price_df['Availability_Status'] == 'Ready_to_Move').astype(int)

# Creating Target Variable

In [ ]:
# Regression Target Variable - Future_Price_5Y
np.random.seed(42)

# Base growth rate by city tier (annual %)
city_growth = House_Price_df.groupby('City')['Price_in_Lakhs'].median()
city_growth_rate = {}

for city in House_Price_df['City'].unique():
    median_price = city_growth.get(city, 250)
    if median_price > 350:
        city_growth_rate[city] = np.random.uniform(0.06, 0.09)
    elif median_price > 200:
        city_growth_rate[city] = np.random.uniform(0.07, 0.11)
    else:
        city_growth_rate[city] = np.random.uniform(0.08, 0.13)
        
House_Price_df['City_Growth_Rate'] = House_Price_df['City'].map(city_growth_rate)


# Property type adjustment
type_multiplier = {'Apartment': 1.0, 'Independent House': 1.05, 'Villa': 1.1}
House_Price_df['Type_Multiplier'] = House_Price_df['Property_Type'].map(type_multiplier)


# New properties, good infrastructure, amenities → higher appreciation
feature_bonus = (
    0.005 * House_Price_df['Is_New_Property'] +
    0.003 * House_Price_df['Has_Security'] +
    0.002 * House_Price_df['Has_Parking'] +
    0.003 * House_Price_df['Is_Ready'] +
    0.001 * House_Price_df['Amenities_Count'] +
    0.005 * (House_Price_df['Infrastructure_Score'] - 0.5)
)


effective_rate = House_Price_df['City_Growth_Rate'] * House_Price_df['Type_Multiplier'] + feature_bonus

# Add small noise for realism
noise = np.random.normal(0, 0.005, len(House_Price_df))
effective_rate = (effective_rate + noise).clip(0.03, 0.18)

# Future Price = Current × (1 + r)^5
House_Price_df['Future_Price_5Y'] = House_Price_df['Price_in_Lakhs'] * (1 + effective_rate) ** 5

print('Future_Price_5Y Created')
print(f"Current Price Range: ₹{House_Price_df['Price_in_Lakhs'].min():.2f}-₹{House_Price_df['Price_in_Lakhs'].max():.2f}")
print(f"Future Price Range: ₹{House_Price_df['Future_Price_5Y'].min():.2f}-₹{House_Price_df['Future_Price_5Y'].max():.2f}")

In [ ]:
# Creating Classification Variable (Multi Factor Score)

# Factor 1: Price below city median → undervalued
city_median_price = House_Price_df.groupby('City')['Price_in_Lakhs'].transform('median')
House_Price_df['Below_City_Median'] = (House_Price_df['Price_in_Lakhs'] <= city_median_price).astype(int)

# Factor 2: Price per sqft below city median → better value
city_median_ppsf = House_Price_df.groupby('City')['Price_per_SqFt'].transform('median')
House_Price_df['Below_City_Median_PPSF'] = (House_Price_df['Price_per_SqFt'] <= city_median_ppsf).astype(int)

# Factor 3: Good infrastructure
House_Price_df['Good_Infrastructure'] = (House_Price_df['Infrastructure_Score'] >= House_Price_df['Infrastructure_Score'].median()).astype(int)

# Factor 4: BHK >= 3 (family-friendly, higher demand)
House_Price_df['Is_Family_Size'] = (House_Price_df['BHK'] >= 3).astype(int)

# Factor 5: Ready to move + Security
House_Price_df['Ready_Secure'] = ((House_Price_df['Is_Ready'] == 1) & (House_Price_df['Has_Security'] == 1)).astype(int)

# Factor 6: High amenities (4+)
House_Price_df['High_Amenities'] = (House_Price_df['Amenities_Count'] >= 4).astype(int)

# Investement Score 
House_Price_df['Investment_Score'] = (
    House_Price_df['Below_City_Median']*2 +
    House_Price_df['Below_City_Median_PPSF']*2 +
    House_Price_df['Good_Infrastructure'] +
    House_Price_df['Is_Family_Size'] +
    House_Price_df['Ready_Secure'] +
    House_Price_df['High_Amenities']
)

threshold = 5
House_Price_df['Good_Investment'] = (House_Price_df['Investment_Score'] >= threshold).astype(int)

# Exploratory Data Analysis

In [ ]:
# 1-> What is the distribution of property prices?
fig, axes = plt.subplots(1, 2, figsize = (14, 5))

axes[0].hist(House_Price_df['Price_in_Lakhs'], bins = 50, color = 'skyblue', edgecolor='white', alpha=0.9)
axes[0].set_xlabel('Price in Lakhs')
axes[0].set_ylabel('Frequency ')
axes[0].set_title('Distribution of Property Prices')
axes[0].axvline(House_Price_df['Price_in_Lakhs'].median(), color='red', linestyle='--',
               label=f"Median: Rs {House_Price_df['Price_in_Lakhs'].median():.0f}L")
axes[0].legend()



axes[1].hist(House_Price_df['Price_in_Lakhs'], bins=50, color='skyblue', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Price in Lakhs')
axes[1].set_ylabel('Frequency (log)')
axes[1].set_title('Price Distribution (Log Scale)')
axes[1].set_yscale('log')
axes[1].axvline(House_Price_df['Price_in_Lakhs'].median(), color='red', linestyle='--',
               label=f"Median: Rs {House_Price_df['Price_in_Lakhs'].median():.0f}L")
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"Price stats: Mean=₹{House_Price_df['Price_in_Lakhs'].mean():.1f}L, Median=₹{House_Price_df['Price_in_Lakhs'].median():.1f}L, Std=₹{House_Price_df['Price_in_Lakhs'].std():.1f}L")

In [ ]:
# 2 -> What is the distribution of property sizes?
fig, axes = plt.subplots(1, 2, figsize = (14, 5))

axes[0].hist(House_Price_df['Size_in_SqFt'], bins = 50, color = 'lightgreen', edgecolor = 'white', alpha = 0.8)
axes[0].set_xlabel('Size in SqFt')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Property Sizes')
axes[0].axvline(House_Price_df['Size_in_SqFt'].median(), color = 'red', linestyle = '--', 
                label = f"Median: {House_Price_df['Size_in_SqFt'].median():.0f}SqFt")
axes[0].legend()

# By BHK
bhk_data = [House_Price_df[House_Price_df['BHK'] == b]['Size_in_SqFt'].values for b in sorted(House_Price_df['BHK'].unique())]
axes[1].boxplot(bhk_data, labels = sorted(House_Price_df['BHK'].unique()))
axes[1].set_xlabel('BHK')
axes[1].set_ylabel('Size (SqFt)')
axes[1].set_title('Property Size by BHK')

plt.tight_layout()
plt.show()

In [ ]:
# How does the price per sq ft vary by property type?
fig, ax = plt.subplots(figsize = (10, 5))
property_type = House_Price_df['Property_Type'].unique()
colors = ['#2196F3', '#FF9800', '#4CAF50']

for i, pt in enumerate(property_type):
    subset = House_Price_df[House_Price_df['Property_Type'] == pt]['Price_per_SqFt']
    ax.hist(subset, bins = 40, alpha = 0.5, label = f"{pt} (median: {subset.median():.4f})", color = colors[i])

ax.set_xlabel('Price per SqFt (₹ Lakhs)')
ax.set_ylabel('Frequency')
ax.set_title('Price per SqFt by Property Type')
ax.legend()
plt.tight_layout()
plt.show()

# Boxplot 
fig, ax = plt.subplots(figsize = (8, 5))
House_Price_df.boxplot(column='Price_per_SqFt', by='Property_Type', ax=ax)
ax.set_title('Price per SqFt by Property Type')
ax.set_ylabel('Price per SqFt (₹ Lakhs)')
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# 4-> Is there a relationship between property size and price?
fig, ax = plt.subplots(figsize = (10, 6))
sample = House_Price_df.sample(5000, random_state = 42)

scatter = ax.scatter(sample['Size_in_SqFt'], sample['Price_in_Lakhs'], c =sample['BHK'],
                    cmap='viridis', alpha=0.6, s=15)
plt.colorbar(scatter, label = 'BHK')
ax.set_xlabel('Size in SqFt')
ax.set_ylabel('Price in Lakhs')
ax.set_title('Property Size vs Price (colored by BHK)')


# corr
corr = House_Price_df['Size_in_SqFt'].corr(House_Price_df['Price_in_Lakhs'])
ax.text(0.05, 0.95, f'Correlation: {corr:.3f}', transform=ax.transAxes, 
        fontsize=12, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat'))

plt.tight_layout()
plt.show()

In [ ]:
# 5 -> Are there any outliers in price per sq ft or property size?
fig, axes = plt.subplots(1, 2, figsize = (14, 5))

# Price per SqFt outliers
bp1 = axes[0].boxplot(House_Price_df['Price_per_SqFt'], vert=True, patch_artist=True)
bp1['boxes'][0].set_facecolor('#FF6B6B')
axes[0].set_title('Price per SqFt - Outlier Check')
axes[0].set_ylabel('Price per SqFt (₹ Lakhs)')
Q1 = House_Price_df['Price_per_SqFt'].quantile(0.25)
Q3 = House_Price_df['Price_per_SqFt'].quantile(0.75)
IQR = Q3 - Q1
outliers_ppsf = ((House_Price_df['Price_per_SqFt'] < Q1 - 1.5*IQR)|(House_Price_df['Price_per_SqFt']> Q3 + 1.5*IQR)).sum()
axes[0].text(0.5, 0.02, f'Outliers: {outliers_ppsf:,}', transform=axes[0].transAxes, 
             ha='center', fontsize=11, color='red')

# Size outliers
bp2 = axes[1].boxplot(House_Price_df['Size_in_SqFt'], vert=True, patch_artist=True)
bp2['boxes'][0].set_facecolor('#4ECDC4')
axes[1].set_title('Property Size - Outlier Check')
axes[1].set_ylabel('Size (SqFt)')

Q1s = House_Price_df['Size_in_SqFt'].quantile(0.25)
Q3s = House_Price_df['Size_in_SqFt'].quantile(0.75)
IQRs = Q3s - Q1s
outliers_size = ((House_Price_df['Size_in_SqFt'] < Q1s - 1.5*IQRs) | (House_Price_df['Size_in_SqFt'] > Q3s + 1.5*IQRs)).sum()
axes[1].text(0.5, 0.02, f'Outliers: {outliers_size:,}', transform=axes[1].transAxes, 
             ha='center', fontsize=11, color='red')

plt.tight_layout()
plt.show()
print(f"Price_per_SqFt outliers (IQR): {outliers_ppsf:,} ({outliers_ppsf/len(House_Price_df)*100:.1f}%)")
print(f"Size_in_SqFt outliers (IQR):   {outliers_size:,} ({outliers_size/len(House_Price_df)*100:.1f}%)")


In [ ]:
# 6-> What is the average price per sq ft by state?
state_ppsf = House_Price_df.groupby('State')['Price_per_SqFt'].mean().sort_values(ascending = True)


fig, ax = plt.subplots(figsize = (12,8))
bars = ax.barh(state_ppsf.index, state_ppsf.values, color=plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(state_ppsf))))
ax.set_xlabel('Avg Price per SqFt (₹ Lakhs)')
ax.set_title('Average Price per SqFt by State')

for bar, val in zip(bars, state_ppsf.values):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# 7 -> Average Property Price by City 
city_price = House_Price_df.groupby('City')['Price_in_Lakhs'].mean().sort_values(ascending = False).head(20)

fig, ax = plt.subplots(figsize = (14,7))
bar = ax.bar(range(len(city_price)), city_price.values, color='steelblue', alpha=0.8)
ax.set_xticks(range(len(city_price)))
ax.set_xticklabels(city_price.index, rotation=45, ha='right')
ax.set_ylabel('Avg Price (₹ Lakhs)')
ax.set_title('Average Property Price by City (Top 20)')


In [ ]:
# 8-> What is the median age of properties by locality?
locality_age = House_Price_df.groupby('Locality')['Age_of_Property'].median().sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize = (14,6))
ax.barh(locality_age.index, locality_age.values, color='salmon', alpha=0.8)
ax.set_xlabel('Median Property Age (years)')
ax.set_title('Median Age of Properties by Locality (Top 20 oldest)')
plt.tight_layout()
plt.show()

In [ ]:
# 9 -> BHK Distribution across Cities
top_cities = House_Price_df['City'].value_counts().head(10).index
df_top = House_Price_df[House_Price_df['City'].isin(top_cities)]

fig, ax = plt.subplots(figsize=(14, 6))
ct = pd.crosstab(df_top['City'], df_top['BHK'], normalize='index') * 100
ct.plot(kind='bar', stacked=True, ax=ax, colormap='Set2', edgecolor='white')
ax.set_ylabel('Percentage (%)')
ax.set_title('Q9: BHK Distribution across Top 10 Cities')
ax.legend(title='BHK', bbox_to_anchor=(1.05, 1))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# 10 -> What are the price trends for the top 5 most expensive localities?
locality_avg = House_Price_df.groupby('Locality')['Price_in_Lakhs'].mean()
top5_localities = locality_avg.nlargest(5).index

df_top5 = House_Price_df[House_Price_df['Locality'].isin(top5_localities)]
locality_year = df_top5.groupby(['Locality', 'Year_Built'])['Price_in_Lakhs'].mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 6))
for loc in top5_localities:
    data = locality_year[locality_year['Locality'] == loc].sort_values('Year_Built')
    ax.plot(data['Year_Built'], data['Price_in_Lakhs'], marker='o', markersize=3, label=loc, alpha=0.8)

ax.set_xlabel('Year Built')
ax.set_ylabel('Avg Price (₹ Lakhs)')
ax.set_title('Price Trends for Top 5 Most Expensive Localities')
ax.legend(bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

In [ ]:
# 11 -> How are numeric features correlated with each other?
numeric_cols = ['Price_in_Lakhs', 'Size_in_SqFt', 'BHK', 'Price_per_SqFt', 
                'Age_of_Property', 'Floor_No', 'Total_Floors', 
                'Nearby_Schools', 'Nearby_Hospitals', 'Amenities_Count',
                'Size_per_BHK', 'Infrastructure_Score']

corr_matrix = House_Price_df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix of Numeric Features')
plt.tight_layout()
plt.show()

In [ ]:
# 12 -> Nearby Schools vs Price per SqFt
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
school_price = House_Price_df.groupby('Nearby_Schools')['Price_per_SqFt'].mean()

axes[0].bar(school_price.index, school_price.values, color='#2196F3', alpha=0.8)
axes[0].set_xlabel('Number of Nearby Schools')
axes[0].set_ylabel('Avg Price Per SqFt')
axes[0].set_title('Nearby Schools vs Price per SqFt')

# 13 ->Nearby Hospitals vs Price per SqFt
hospital_price = House_Price_df.groupby('Nearby_Hospitals')['Price_per_SqFt'].mean()
axes[1].bar(hospital_price.index, hospital_price.values, color = '#FF5722', alpha = 0.8)
axes[1].set_xlabel('Number of Nearby Hospital')
axes[1].set_ylabel('Avg Price Per SqFt')
axes[1].set_title('Nearby Hospital VS Price Per SqFt')
plt.tight_layout()
plt.show()

In [ ]:
# 14 -> Price by Furnished Status
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

furnished_price = House_Price_df.groupby('Furnished_Status')['Price_in_Lakhs'].agg(['mean', 'median'])
x = range(len(furnished_price))
axes[0].bar(x, furnished_price['mean'], width=0.35, label='Mean', color='steelblue', alpha=0.8)
axes[0].bar([i+0.35 for i in x], furnished_price['median'], width=0.35, label='Median', color='coral', alpha=0.8)
axes[0].set_xticks([i+0.175 for i in x])
axes[0].set_xticklabels(furnished_price.index)
axes[0].set_ylabel('Price (₹ Lakhs)')
axes[0].set_title('Price by Furnished Status')
axes[0].legend()

# 15 -> Price per SqFt by Facing Direction 
facing_price = House_Price_df.groupby('Facing')['Price_per_SqFt'].agg(['mean', 'median'])
x = range(len(facing_price))
axes[1].bar(x, facing_price['mean'], width=0.35, label='Mean', color='#4CAF50', alpha=0.8)
axes[1].bar([i+0.35 for i in x], facing_price['median'], width=0.35, label='Median', color='#FFC107', alpha=0.8)
axes[1].set_xticks([i+0.175 for i in x])
axes[1].set_xticklabels(facing_price.index)
axes[1].set_ylabel('Price per SqFt')
axes[1].set_title('Q15: Price per SqFt by Facing Direction')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 16 -> Properties by Owner Type 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

owner_counts = House_Price_df['Owner_Type'].value_counts()
colors_pie = ['#2196F3', '#FF9800', '#4CAF50']
axes[0].pie(owner_counts, labels=owner_counts.index, autopct='%1.1f%%', colors=colors_pie, startangle=90)
axes[0].set_title('Properties by Owner Type')

# 17 -> Properties by Availability Status 
avail_counts = House_Price_df['Availability_Status'].value_counts()
axes[1].pie(avail_counts, labels=avail_counts.index, autopct='%1.1f%%', colors=['#4CAF50', '#FF5722'], startangle=90)
axes[1].set_title('Properties by Availability Status')

plt.tight_layout()
plt.show()

In [ ]:
# 18 -> Does Parking Space affect Property Price? 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

parking_price = House_Price_df.groupby('Parking_Space')['Price_in_Lakhs'].agg(['mean', 'median'])
parking_price.plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], alpha=0.8)
axes[0].set_title('Parking Space vs Price')
axes[0].set_ylabel('Price (₹ Lakhs)')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# 19 -> Amenities Count vs Price per SqFt 
amenity_ppsf = House_Price_df.groupby('Amenities_Count')['Price_per_SqFt'].agg(['mean', 'median'])
amenity_ppsf.plot(kind='bar', ax=axes[1], color=['#9C27B0', '#FF9800'], alpha=0.8)
axes[1].set_title('Amenities Count vs Price per SqFt')
axes[1].set_ylabel('Price per SqFt')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# 20 -> Public Transport Accessibility vs Price per SqFt & Investment 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Price per SqFt by Transport
transport_order = ['Low', 'Medium', 'High']
transport_ppsf = House_Price_df.groupby('Public_Transport_Accessibility')['Price_per_SqFt'].mean().reindex(transport_order)
axes[0].bar(transport_ppsf.index, transport_ppsf.values, color=['#FF5722', '#FFC107', '#4CAF50'], alpha=0.8)
axes[0].set_ylabel('Avg Price per SqFt')
axes[0].set_title('Transport Accessibility vs Price per SqFt')

# Good Investment % by Transport
transport_invest = House_Price_df.groupby('Public_Transport_Accessibility')['Good_Investment'].mean().reindex(transport_order) * 100
axes[1].bar(transport_invest.index, transport_invest.values, color=['#FF5722', '#FFC107', '#4CAF50'], alpha=0.8)
axes[1].set_ylabel('Good Investment %')
axes[1].set_title('Transport Accessibility vs Good Investment Rate')
for i, (idx, val) in enumerate(transport_invest.items()):
    axes[1].text(i, val + 0.5, f'{val:.1f}%', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

# Outlier Detection and Treatment 

In [ ]:
outlier_cols = ['Price_in_Lakhs', 'Size_in_SqFt', 'Price_per_SqFt']

for col in outlier_cols:
    Q1 = House_Price_df[col].quantile(0.25)
    Q3 = House_Price_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    before = ((House_Price_df[col] < lower) | (House_Price_df[col] > upper)).sum()
    House_Price_df[col] = House_Price_df[col].clip(lower, upper)
    after = ((House_Price_df[col] < lower) | (House_Price_df[col] > upper)).sum()
    
    print(f"  {col}: {before:,} outliers capped (lower={lower:.2f}, upper={upper:.2f})")
    
House_Price_df['Future_Price_5Y'] = House_Price_df['Price_in_Lakhs'] * (1 + effective_rate) ** 5

print("\n✅ Outliers treated. Recalculated Future_Price_5Y.")

# Data Preprocessing for Machine Learning 

In [ ]:
# Dropping Col which Help in Target Creation 
drop_cols = [
    'Locality', 'Amenities',
    'Type_Multiplier', 'City_Growth_Rate',
    'Below_City_Median_PPSF', 'Below_City_Median', 'Good_Infrastructure',
    'Is_Family_Size', 'Ready_Secure', 'High_Amenities', 'Investment_Score',
    'Good_Investment', 'Future_Price_5Y'
]

# Taking Target
y_reg = House_Price_df['Future_Price_5Y'].copy()
y_class = House_Price_df['Good_Investment'].copy()

House_Price_df_features = House_Price_df.drop(columns = [c for c in drop_cols if col in House_Price_df.columns],
                                              errors = 'ignore')
print(f'Shape of Dataset: {House_Price_df_features.shape}')
print(f'Existing Columns: {House_Price_df_features.columns}')

In [ ]:
# Lable Encoding 
label_encoders = {}

# One-hot encode: State, City, Property_Type, Furnished_Status, Facing, Owner_Type
cat_cols_to_encode = ['State', 'City', 'Property_Type', 'Furnished_Status',
                      'Public_Transport_Accessibility', 'Parking_Space', 'Security',
                      'Facing', 'Owner_Type', 'Availability_Status']

cat_cols_to_encode = [c for c in cat_cols_to_encode if c in House_Price_df_features.columns]

print("Encoding categorical columns:", cat_cols_to_encode)

for col in cat_cols_to_encode:
    le = LabelEncoder()
    House_Price_df_features[col] = le.fit_transform(House_Price_df_features[col])
    label_encoders[col] = le
    print(f"  {col}: {len(le.classes_)} classes → encoded")

print(f"\n✅ All categorical features encoded. Shape: {House_Price_df_features.shape}")


In [ ]:
# Scaling the Feature 
scaler = RobustScaler()

binary_cols = [c for c in House_Price_df_features.columns if House_Price_df_features[c].nunique() <= 2]
scale_cols = [c for c in House_Price_df_features.columns if c not in binary_cols]

X = House_Price_df_features.copy()
X[scale_cols] = scaler.fit_transform(X[scale_cols])

print(f"\n✅ Features scaled. Final X shape: {X.shape}")
print(f"Regression target (y_reg) shape: {y_reg.shape}")
print(f"Classification target (y_clf) shape: {y_class.shape}")
print(f"  Good Investment distribution: {y_class.value_counts().to_dict()}")

In [ ]:
# Train Test Split
# Split for Regression 
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size = 0.20, random_state = 42)

# Split for Classification
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X, y_class, test_size = 0.20, random_state = 42,
                                                                   stratify = y_class)

print(f"Regression -> Train: {X_train_reg.shape[0]:,}, Test: {X_test_reg.shape[0]:,}")
print(f"Clasification -> Train: {X_train_clf.shape[0]:,}, Test: {X_test_clf.shape[0]:,}")

# Model Deployment ( Regression )

In [ ]:
regression_models = {
    'Linear Regression' : LinearRegression(),
    'Ridge Regression' : Ridge(alpha = 1.0, random_state = 42),
    'Lasso Regression' : Lasso(alpha = 1.0, random_state = 42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=6, random_state=42)
}

reg_result = {}

print('Regression Models - Predicting Future_Price_5Y')
print('-'*70)

for name, model in regression_models.items():
    print(f"Training {name}")
    model.fit(X_train_reg, y_train_reg)
    
    y_pred_train = model.predict(X_train_reg)
    y_pred_test = model.predict(X_test_reg)
    
    results = {
        'model' : model,
        'train_r2' : r2_score(y_train_reg, y_pred_train),
        'test_r2' : r2_score(y_test_reg, y_pred_test),
        'RMSE' : np.sqrt(mean_squared_error(y_test_reg, y_pred_test)),
        'MAE' : mean_absolute_error(y_test_reg, y_pred_test),
        'y_pred_test' : y_pred_test
    }
    
    reg_result[name] =results
    
    print(f"  Train R²: {results['train_r2']:.4f}")
    print(f"  Test R²:  {results['test_r2']:.4f}")
    print(f"  RMSE:     {results['RMSE']:.2f}")
    print(f"  MAE:      {results['MAE']:.2f}")

In [ ]:
# Regression Summary 

reg_summary = pd.DataFrame({
    name: {
        'Train R²': r['train_r2'],
        'Test R²': r['test_r2'],
        'RMSE': r['RMSE'],
        'MAE': r['MAE'],
    }
    for name, r in reg_result.items()
}).T.sort_values('Test R²', ascending=False)

print("\n── Regression Model Comparison ──")
print(reg_summary.round(4).to_string())

best_reg_name = reg_summary['Test R²'].idxmax()
print(f"\n🏆 Best Regression Model: {best_reg_name} (Test R² = {reg_summary.loc[best_reg_name, 'Test R²']:.4f})")

# Model Deployment ( Classification )

In [ ]:
# Classification Model 

classification_models = {
    'Logistic Regression' : LogisticRegression(random_state=42, max_iter=1000, n_jobs=-1),
    'Random Forest' : RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting' : GradientBoostingClassifier(n_estimators=100, max_depth=6, random_state=42),
    'XGBoost' : XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, 
                              n_jobs=-1, verbosity=0, eval_metric='logloss'),
    'KNN' : KNeighborsClassifier(n_neighbors=7, n_jobs=-1)
}

clf_result ={}

print("Classification Model - Predicting Good_Investment")
print('-'*70)

for name, model in classification_models.items():
    print(f"Training {name}..")
    model.fit(X_train_clf, y_train_clf)
    
    y_pred_train = model.predict(X_train_clf)
    y_pred_test = model.predict(X_test_clf)
    
    if hasattr(model, 'predict_proba'):
        y_proba_test = model.predict_proba(X_test_clf)[:, 1]
    else:
        y_proba_test = model.decision_function(X_test_clf)
        
    results = {
        'model': model,
        'train_acc': accuracy_score(y_train_clf, y_pred_train),
        'test_acc': accuracy_score(y_test_clf, y_pred_test),
        'precision': precision_score(y_test_clf, y_pred_test),
        'recall': recall_score(y_test_clf, y_pred_test),
        'f1': f1_score(y_test_clf, y_pred_test),
        'roc_auc': roc_auc_score(y_test_clf, y_proba_test),
        'y_pred_test': y_pred_test,
        'y_proba_test': y_proba_test,
    }
    clf_result[name] = results
    
    print(f"  Train Acc: {results['train_acc']:.4f}")
    print(f"  Test Acc:  {results['test_acc']:.4f}")
    print(f"  Precision: {results['precision']:.4f}")
    print(f"  Recall:    {results['recall']:.4f}")
    print(f"  F1 Score:  {results['f1']:.4f}")
    print(f"  ROC AUC:   {results['roc_auc']:.4f}")

In [ ]:
clf_summary = pd.DataFrame({
    name: {
        'Train Acc': r['train_acc'],
        'Test Acc': r['test_acc'],
        'Precision': r['precision'],
        'Recall': r['recall'],
        'F1 Score': r['f1'],
        'ROC AUC': r['roc_auc'],
    }
    for name, r in clf_result.items()
}).T.sort_values('Test Acc', ascending=False)

print("\n── Classification Model Comparison ──")
print(clf_summary.round(4).to_string())

best_clf_name = clf_summary['Test Acc'].idxmax()
print(f"\n🏆 Best Classification Model: {best_clf_name} (Test Acc = {clf_summary.loc[best_clf_name, 'Test Acc']:.4f})")

In [ ]:
# Model Performance Comparision 
fig, axes = plt.subplots(1, 2, figsize = (15,6))

reg_names = reg_summary.index.tolist()
x = range(len(reg_names))
axes[0].bar([i - 0.15 for i in x], reg_summary['Train R²'], width=0.3, label='Train R²', color='steelblue', alpha=0.8)
axes[0].bar([i + 0.15 for i in x], reg_summary['Test R²'], width=0.3, label='Test R²', color='coral', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(reg_names, rotation=45, ha='right')
axes[0].set_ylabel('R² Score')
axes[0].set_title('Regression: R² Score Comparison')
axes[0].legend()
axes[0].set_ylim(0, 1.05)

# RMSE Comparison
axes[1].bar(reg_names, reg_summary['RMSE'], color='#FF6B6B', alpha=0.8)
axes[1].set_xticklabels(reg_names, rotation=45, ha='right')
axes[1].set_ylabel('RMSE')
axes[1].set_title('Regression: RMSE Comparison (lower is better)')

plt.tight_layout()
plt.show()

In [ ]:
# ── Classification Performance Comparison ──
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

clf_names = clf_summary.index.tolist()
x = range(len(clf_names))

# Accuracy/F1/AUC Comparison
metrics_to_plot = ['Test Acc', 'F1 Score', 'ROC AUC']
width = 0.25
for i, metric in enumerate(metrics_to_plot):
    axes[0].bar([j + i*width - width for j in x], clf_summary[metric], width=width, label=metric, alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(clf_names, rotation=45, ha='right')
axes[0].set_ylabel('Score')
axes[0].set_title('Classification: Model Comparison')
axes[0].legend()
axes[0].set_ylim(0, 1.05)

# Precision vs Recall
axes[1].bar([i - 0.15 for i in x], clf_summary['Precision'], width=0.3, label='Precision', color='#4CAF50', alpha=0.8)
axes[1].bar([i + 0.15 for i in x], clf_summary['Recall'], width=0.3, label='Recall', color='#FF9800', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(clf_names, rotation=45, ha='right')
axes[1].set_ylabel('Score')
axes[1].set_title('Classification: Precision vs Recall')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# ── Confusion Matrix for Best Classifier ──
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
best_clf = clf_result[best_clf_name]
ConfusionMatrixDisplay.from_predictions(
    y_test_clf, best_clf['y_pred_test'], 
    display_labels=['Not Good', 'Good Investment'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title(f'Confusion Matrix: {best_clf_name}')

# ROC Curve for all models
for name, r in clf_result.items():
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_test_clf, r['y_proba_test'])
    axes[1].plot(fpr, tpr, label=f"{name} (AUC={r['roc_auc']:.3f})")
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves — All Classification Models')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance (Best Models)
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Regression Important Feature
best_reg_model = reg_result[best_reg_name]['model']
if hasattr(best_reg_model, 'feature_importances_'):
    feat_imp_reg = pd.Series(best_reg_model.feature_importances_, index = X.columns).nlargest(15)
    feat_imp_reg.sort_values().plot(kind='barh', ax=axes[0], color='steelblue', alpha=0.8)
    axes[0].set_title(f'Feature Importance: {best_reg_name} (Regression)')
    
elif hasattr(best_reg_model, 'coef_'):
    feat_imp_reg = pd.Series(np.abs(best_reg_model.coef_), index=X.columns).nlargest(15)
    feat_imp_reg.sort_values().plot(kind='barh', ax=axes[0], color='steelblue', alpha=0.8)
    axes[0].set_title(f'Feature Importance (|coef|): {best_reg_name} (Regression)')
    
    
best_clf_model = clf_result[best_clf_name]['model']
if hasattr(best_clf_model, 'feature_importances_'):
    feat_imp_clf = pd.Series(best_clf_model.feature_importances_, index=X.columns).nlargest(15)
    feat_imp_clf.sort_values().plot(kind='barh', ax=axes[1], color='coral', alpha=0.8)
    axes[1].set_title(f'Feature Importance: {best_clf_name} (Classification)')

plt.tight_layout()
plt.show()

In [ ]:
# ML Flow Experiment Tracking 

mlflow.set_tracking_uri('mlflow_runs')
mlflow.set_experiment("Real_Estate_Investment_Advisor")

print(" Logging Regression Model to ML Flow")

for name, r in reg_result.items():
    with mlflow.start_run(run_name=f"Reg_{name.replace(' ', '_')}"):
        mlflow.log_param("model_type", "regression")
        mlflow.log_param("model_name", name)
        mlflow.log_param("test_size", 0.2)
        mlflow.log_metric("train_r2", r['train_r2'])
        mlflow.log_metric("test_r2", r['test_r2'])
        mlflow.log_metric("RMSE", r['RMSE'])
        mlflow.log_metric("MAE", r['MAE'])
        mlflow.sklearn.log_model(r['model'], name = f"model_{name.replace(' ', '_')}")
        print(f"  ✅ Logged {name}: R²={r['test_r2']:.4f}, RMSE={r['RMSE']:.2f}")

        
# ── Log Classification Models ──
print("\n── Logging Classification Models to MLflow ──\n")
for name, r in clf_result.items():
    with mlflow.start_run(run_name=f"Clf_{name.replace(' ', '_')}"):
        mlflow.log_param("model_type", "classification")
        mlflow.log_param("model_name", name)
        mlflow.log_param("test_size", 0.2)
        mlflow.log_metric("accuracy", r['test_acc'])
        mlflow.log_metric("precision", r['precision'])
        mlflow.log_metric("recall", r['recall'])
        mlflow.log_metric("f1_score", r['f1'])
        mlflow.log_metric("roc_auc", r['roc_auc'])
        mlflow.sklearn.log_model(r['model'], name = f"model_{name.replace(' ', '_')}")
        print(f"  ✅ Logged {name}: Acc={r['test_acc']:.4f}, AUC={r['roc_auc']:.4f}")

print("\n🏆 All models logged to MLflow successfully!")

# Saving Best Model and Artifact

In [ ]:
# Saving Best Model
joblib.dump(reg_result[best_reg_name]['model'], 'best_regression_model.pkl')
joblib.dump(clf_result[best_clf_name]['model'], 'best_classification_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(label_encoders, 'label_encoders.pkl')

joblib.dump(list(X.columns), 'feature_columns.pkl')

print(f"✅ Saved best regression model: {best_reg_name}")
print(f"✅ Saved best classification model: {best_clf_name}")
print(f"✅ Saved scaler, label encoders, and feature columns")

In [ ]:
# ── Save cleaned dataset ──
df_export = House_Price_df.copy()
df_export.to_csv('india_housing_prices_cleaned.csv', index=False)
print(f"✅ Cleaned dataset saved: {df_export.shape[0]:,} rows × {df_export.shape[1]} columns")

# StreamLit Application Deployment

In [1]:
%%writefile steamlit_real_estate.py
'''
Real Estate Investment Advisor
Streamlit Application - Predicting Property Profitability & Future Value
'''

import streamlit as st
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
from sklearn.metrics import roc_curve, auc
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Page Configuration 
st.set_page_config(
    page_title="🏠 Real Estate Investment Advisor",
    page_icon="🏠",
    layout="wide"
)

st.markdown('''
<style>
    .main-header{
        font-size: 1.0rem;
        color: #2E86AB;
        text-align: center;
        margin-bottom: 1rem;
    }
    .sub-header {
        font-size: 1.5rem;
        color: #264653;
        margin-top: 1rem;
        margin-bottom: 1rem;
    }
    .metric-card {
        background-color: #f8f9fa;
        border-radius: 10px;
        padding: 20px;
        box-shadow: 0 4px 6px rgba(0,0,0,0.1);
        margin-bottom: 20px;
    }
    .good-investment {
        background-color: #d4edda;
        border-left: 5px solid #28a745;
        padding: 15px;
        border-radius: 5px;
    }
    .bad-investment {
        background-color: #fff3cd;
        border-left: 5px solid #ffc107;
        padding: 15px;
        border-radius: 5px;
    }
</style>
''', unsafe_allow_html = True)

# Title
st.markdown("<h1 class = 'main-header'> 🏠 Real Estate Investment Advisor</h1>", unsafe_allow_html = True)
st.markdown("<h3 class = 'sub-header'> ML-powered Property Investment Analysis & Future Price Prediction </h3>",
            unsafe_allow_html = True)

# Load Model, Artifact and Data
@st.cache_resource
def load_artifacts():
    try:
        reg_model = joblib.load('best_regression_model.pkl')
        clf_model = joblib.load('best_classification_model.pkl')
        scaler = joblib.load('scaler.pkl')
        label_encoders = joblib.load('label_encoders.pkl')
        feature_columns = joblib.load('feature_columns.pkl')
        
        try:
            House_Price_df = pd.read_csv('india_housing_prices_cleaned.csv')
        except:
            House_Price_df = None
        
        return reg_model, clf_model, scaler, label_encoders, feature_columns, House_Price_df
    
    except Exception as e:
        st.error(f"Error loading artifacts: {e}")
        return None, None, None, None, None, None

reg_model, clf_model, scaler, label_encoders, feature_columns, df_clean = load_artifacts()

# Sidebar Navigation 
st.sidebar.header("📊 Navigation")
page = st.sidebar.radio("Go to:", ["🏠 Property Analysis", "📈 Market Insights", "🔍 Data Explorer", "⚙️ Model Info"])
    
# Buildung Helper Function 
def prepare_features(input_data):
    
    input_df = pd.DataFrame([input_data])
    
    # For Label Encoder
    for col in label_encoders.keys():
        if col in input_df.columns:
            try:
                input_df[col] = label_encoders[col].transform(input_df[col].astype(str))
            except ValueError:
                input_df[col] = 0
                
    for col in feature_columns:
        if col not in input_df.columns:
            input_df[col] = 0
    
    input_df = input_df[feature_columns]
    
    # Scale Feature
    if scaler is not None:
        binary_cols = [c for c in input_df.columns if input_df[c].nunique() <= 2]
        scale_cols = [c for c in input_df.columns if c not in binary_cols]
        
        if len(scale_cols) > 0:
            try:
                input_df_scaled = input_df.copy()
                input_df_scaled[scale_cols] = scaler.transform(input_df[scale_cols])
                return input_df_scaled
            
            except:
                return input_df
        
    return input_df

def get_feature_importance(model, feature_names, top_n=10):
    
    # Get Feature importance from model 
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
        indices = np.argsort(importances)[-top_n:][::-1]
        
        return pd.DataFrame({
            'Feature': [feature_names[i] for i in indices],
            'Importance': [importances[i] for i in indices]
        })
    
    elif hasattr(model, 'coef_'):
        importances = np.abs(model.coef_)
        indices = np.argsort(importances)[-top_n:][::-1]
        return pd.DataFrame({
            'Feature': [feature_names[i] for i in indices],
            'Importance': [importances[i] for i in indices]
        })
    return None
# ---------------------------Page 1: Property Analysis Start ----------------------------

if page == "🏠 Property Analysis":
    st.sidebar.header("📋 Property Details")
    
    with st.sidebar.form("property_form"):
        st.subheader("📍 Location")
        
        state = st.selectbox("State", sorted([
            'Andhra Pradesh', 'Assam', 'Bihar', 'Chhattisgarh', 'Delhi',
            'Gujarat', 'Haryana', 'Jharkhand', 'Karnataka', 'Kerala',
            'Madhya Pradesh', 'Maharashtra', 'Odisha', 'Punjab',
            'Rajasthan', 'Tamil Nadu', 'Telangana', 'Uttar Pradesh',
            'Uttarakhand', 'West Bengal'
        ]))
        
        city = st.selectbox("City", sorted([
            'Ahmedabad', 'Amritsar', 'Bangalore', 'Bhopal', 'Bhubaneswar',
            'Bilaspur', 'Chennai', 'Coimbatore', 'Cuttack', 'Dehradun',
            'Delhi', 'Durgapur', 'Dwarka', 'Faridabad', 'Gaya',
            'Guwahati', 'Haridwar', 'Hyderabad', 'Indore', 'Jaipur',
            'Jamshedpur', 'Jodhpur', 'Kochi', 'Kolkata', 'Lucknow',
            'Ludhiana', 'Mangalore', 'Mumbai', 'Mysore', 'Nagpur',
            'New Delhi', 'Noida', 'Patna', 'Pune', 'Raipur',
            'Ranchi', 'Silchar', 'Surat', 'Trivandrum', 'Vijayawada',
            'Vishakhapatnam', 'Warangal'
        ]))
        
        st.subheader("🏠 Property Details")
        col1, col2 = st.columns(2)
        
        with col1:
            property_type = st.selectbox("Property Type", ['Apartment', 'Independent House', 'Villa'])
            bhk = st.slider("BHK", 1, 5, 3)
        with col2:
            size_sqft = st.number_input("Size (SqFt)", min_value=500, max_value=5000, value=2500, step=100)
            current_price = st.number_input("Current Price (₹ Lakhs)", min_value=10.0, 
                                            max_value=500.0, value=250.0, step=10.0)
            
        st.subheader("📅 Construction Details")
        year_built = st.slider("Year Built", 1990, 2023, 2010)
        age_of_property = 2024 - year_built
        
        st.subheader("🛠️ Features")
        col1, col2 = st.columns(2)
        
        with col1:
            furnished_status = st.selectbox("Furnished Status", ['Furnished', 'Semi-furnished', 'Unfurnished'])
            floor_no = st.slider("Floor Number", 0, 30, 10)
            
        with col2:
            total_floors = st.slider("Total Floors", 1, 30, 15)
            
        
        st.subheader("🏫 Infrastructure")
        col1, col2, col3 = st.columns(3)
        with col1:
            nearby_schools = st.slider("Nearby Schools", 1, 10, 5)
        with col2:
            nearby_hospitals = st.slider("Nearby Hospitals", 1, 10, 5)
        with col3:
            transport_access = st.selectbox("Public Transport", ['Low', 'Medium', 'High'])
            
        st.subheader("🚗 Amenities")
        col1, col2 = st.columns(2)
        with col1:
            parking = st.radio("Parking Space", ['Yes', 'No'])
            facing = st.selectbox("Facing Direction", ['North', 'South', 'East', 'West'])
        with col2:
            security = st.radio("Security", ['Yes', 'No'])
            
        st.subheader("👤 Ownership")
        col1, col2 = st.columns(2)
        with col1:
            owner_type = st.selectbox("Owner Type", ['Owner', 'Builder', 'Broker'])
        with col2:
            availability = st.radio("Availability", ['Ready_to_Move', 'Under_Construction'])
            
        st.subheader("🎯 Amenities Checklist")
        col1, col2, col3 = st.columns(3)
        with col1:
            has_gym = st.checkbox("Gym")
            has_pool = st.checkbox("Swimming Pool")
        with col2:
            has_garden = st.checkbox("Garden")
            has_playground = st.checkbox("Playground")
        with col3:
            has_clubhouse = st.checkbox("Clubhouse")
        
        submit_button = st.form_submit_button("🔍 Analyze Property")
        
    if submit_button:
        with st.spinner("Analyzing property..."):
            price_per_sqft = current_price / size_sqft if size_sqft > 0 else 0
            amenities_count = sum([has_gym, has_pool, has_garden, has_playground, has_clubhouse])
            size_per_bhk = size_sqft / bhk if bhk > 0 else size_sqft
            floor_ratio = floor_no / total_floors if total_floors > 0 else 0
            is_high_floor = 1 if floor_ratio > 0.7 else 0
            is_new_property = 1 if age_of_property <= 5 else 0
            is_old_property = 1 if age_of_property > 25 else 0
            school_density = nearby_schools / 10.0
            hospital_density = nearby_hospitals / 10.0
            
            transport_map = {'High': 3, 'Medium': 2, 'Low': 1}
            transport_score = transport_map.get(transport_access, 1)
            infrastructure_score = (school_density + hospital_density + transport_score/3.0) / 3.0
            
            has_parking_bin = 1 if parking == 'Yes' else 0
            has_security_bin = 1 if security == 'Yes' else 0
            is_ready = 1 if availability == 'Ready_to_Move' else 0
            
            # Input Data 
            input_data = {
                'State': state, 'City': city, 'Property_Type': property_type,
                'BHK': bhk, 'Size_in_SqFt': size_sqft, 'Price_in_Lakhs': current_price,
                'Price_per_SqFt': price_per_sqft, 'Year_Built': year_built,
                'Furnished_Status': furnished_status, 'Floor_No': floor_no,
                'Total_Floors': total_floors, 'Age_of_Property': age_of_property,
                'Nearby_Schools': nearby_schools, 'Nearby_Hospitals': nearby_hospitals,
                'Public_Transport_Accessibility': transport_access, 'Parking_Space': parking,
                'Security': security, 'Facing': facing, 'Owner_Type': owner_type,
                'Availability_Status': availability, 'Amenities_Count': amenities_count,
                'Has_Gym': int(has_gym), 'Has_Pool': int(has_pool),
                'Has_Garden': int(has_garden), 'Has_Playground': int(has_playground),
                'Has_Clubhouse': int(has_clubhouse), 'Size_per_BHK': size_per_bhk,
                'Floor_Ratio': floor_ratio, 'Is_High_Floor': is_high_floor,
                'Is_New_Property': is_new_property, 'Is_Old_Property': is_old_property,
                'School_Density_Score': school_density, 'Hospital_Density_Score': hospital_density,
                'Transport_Score': transport_score, 'Infrastructure_Score': infrastructure_score,
                'Has_Parking': has_parking_bin, 'Has_Security': has_security_bin,
                'Is_Ready': is_ready
            }
            
            prepared_features = prepare_features(input_data)
            
            try:
                future_price = reg_model.predict(prepared_features)[0]
                investment_prob = clf_model.predict_proba(prepared_features)[0][1]
                investment_class = clf_model.predict(prepared_features)[0]
                
                appreciation_percent = ((future_price - current_price) / current_price) * 100
                annual_growth = ((future_price / current_price) ** (1/5) - 1) * 100
                
                st.success("✅ Analysis Complete!")
                
            #---------------Result--------------------
                st.markdown("## 📊 Investment Analysis Results")
                # Key Metrics Cards
                col1, col2, col3, col4 = st.columns(4)
                with col1:
                    st.markdown('<div class="metric-card">', unsafe_allow_html=True)
                    st.metric("Current Price", f"₹{current_price:.1f}L")
                    st.markdown('</div>', unsafe_allow_html=True)
                    
                with col2:
                    st.markdown('<div class="metric-card">', unsafe_allow_html=True)
                    st.metric("5-Year Price", f"₹{future_price:.1f}L", f"{appreciation_percent:.1f}%")
                    st.markdown('</div>', unsafe_allow_html=True)
                    
                with col3:
                    st.markdown('<div class="metric-card">', unsafe_allow_html=True)
                    st.metric("Annual Growth", f"{annual_growth:.1f}%", "CAGR")
                    st.markdown('</div>', unsafe_allow_html=True)
                
                
                with col4:
                    st.markdown('<div class="metric-card">', unsafe_allow_html=True)
                    confidence_color = "🟢" if investment_prob > 0.7 else "🟡" if investment_prob > 0.5 else "🔴"
                    st.metric("Confidence", f"{investment_prob*100:.1f}%", confidence_color)
                    st.markdown('</div>', unsafe_allow_html=True)
                    
                # Investment Recommendation
                st.markdown("## 🎯 Investment Recommendation")
                
                if investment_class == 1:
                    st.markdown('<div class="good-investment">', unsafe_allow_html=True)
                    st.success(f"### 🟢 **GOOD INVESTMENT**")
                    st.write(f"**Confidence:** {investment_prob*100:.1f}% | **Probability:** {investment_prob:.3f}")
                    st.write("✅ Strong fundamentals  ✅ Good value proposition  ✅ High growth potential")
                    st.markdown('</div>', unsafe_allow_html=True)
                    
                else:
                    st.markdown('<div class="average-investment">', unsafe_allow_html=True)
                    st.warning(f"### 🟡 **AVERAGE INVESTMENT**")
                    st.write(f"**Confidence:** {investment_prob*100:.1f}% | **Probability:** {investment_prob:.3f}")
                    st.write("⚠️ Consider alternatives  ⚠️ Negotiate price  ⚠️ Evaluate carefully")
                    st.markdown('</div>', unsafe_allow_html=True)
                    
                # --------------------Visual Insight----------------------------------
                
                col1, col2 = st.columns(2)
                
                with col1:
                    # Price evolution chart
                    years = [0, 1, 2, 3, 4, 5]
                    prices = [current_price]
                    for i in range(1, 6):
                        prices.append(current_price * ((1 + annual_growth/100) ** i))
                    
                    fig1 = go.Figure()
                    fig1.add_trace(go.Scatter(x=years, y=prices, mode='lines+markers',
                                             name='Projected Price', line=dict(color='#4CAF50', width=3),
                                             marker=dict(size=8)))
                    fig1.update_layout(title='📊 Price Projection Over 5 Years',
                                      xaxis_title='Years',
                                      yaxis_title='Price (₹ Lakhs)',
                                      template='plotly_white')
                    st.plotly_chart(fig1, use_container_width=True)
                
                with col2:
                    # Feature importance
                    fi_reg = get_feature_importance(reg_model, feature_columns, top_n=10)
                    if fi_reg is not None:
                        fig2 = px.bar(fi_reg, x='Importance', y='Feature', orientation='h',
                                     title='🔍 Top 10 Features Affecting Future Price',
                                     color='Importance', color_continuous_scale='Blues')
                        fig2.update_layout(template='plotly_white', height=400)
                        st.plotly_chart(fig2, use_container_width=True)
                        
                        
                        
                        
                col1, col2 = st.columns(2)
                
                with col1:
                    # Investment factors radar chart
                    categories = ['Price/Value', 'Infrastructure', 'Amenities', 'Location', 'Readiness']
                    values = [
                        min(100, max(0, 100 - (price_per_sqft * 1000))),  # Price per SqFt
                        infrastructure_score * 100,  # Infrastructure
                        min(100, amenities_count * 20),  # Amenities
                        70 if city in ['Bangalore', 'Mumbai', 'Delhi', 'Hyderabad'] else 50,  # Location
                        100 if is_ready == 1 else 60  # Readiness
                    ]
                    
                    fig3 = go.Figure(data=go.Scatterpolar(
                        r=values,
                        theta=categories,
                        fill='toself',
                        name='Property Score',
                        line_color='#2196F3'
                    ))
                    fig3.update_layout(
                        polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
                        title='📊 Property Score Radar Chart',
                        template='plotly_white',
                        height=400
                    )
                    st.plotly_chart(fig3, use_container_width=True)
                
                with col2:
                    # Market comparison (if data available)
                    if df_clean is not None:
                        city_avg = df_clean[df_clean['City'] == city]['Price_in_Lakhs'].mean() if city in df_clean['City'].values else current_price
                        type_avg = df_clean[df_clean['Property_Type'] == property_type]['Price_in_Lakhs'].mean()
                        
                        comparison_data = pd.DataFrame({
                            'Category': ['This Property', f'{city} Avg', f'{property_type} Avg'],
                            'Price (₹L)': [current_price, city_avg, type_avg]
                        })
                        
                        fig4 = px.bar(comparison_data, x='Category', y='Price (₹L)',
                                     title='🏙️ Price Comparison with Market Averages',
                                     color='Category', text='Price (₹L)',
                                     color_discrete_map={'This Property': '#4CAF50', 
                                                         f'{city} Avg': '#2196F3',
                                                         f'{property_type} Avg': '#FF9800'})
                        fig4.update_traces(texttemplate='₹%{y:.0f}L', textposition='outside')
                        fig4.update_layout(template='plotly_white', height=400)
                        st.plotly_chart(fig4, use_container_width=True)
                    
                st.markdown("## 🔍 Detailed Breakdown")
                
                # Property Scorecard
                col1, col2, col3 = st.columns(3)
                
                with col1:
                    st.markdown("### 💰 Price Analysis")
                    st.metric("Price per SqFt", f"₹{price_per_sqft:.3f}L", 
                             "Good" if price_per_sqft < 0.09 else "Average")
                    st.metric("Size per BHK", f"{size_per_bhk:.0f} sqft")
                    st.metric("Value Score", f"{(100 - price_per_sqft*1000):.0f}/100")
                
                with col2:
                    st.markdown("### 🏗️ Infrastructure")
                    st.metric("Infrastructure Score", f"{infrastructure_score:.2f}/1.0")
                    st.metric("Schools", nearby_schools, f"Density: {school_density:.1f}")
                    st.metric("Hospitals", nearby_hospitals, f"Density: {hospital_density:.1f}")
                    st.metric("Transport", transport_access, f"Score: {transport_score}")
                
                with col3:
                    st.markdown("### 🎯 Features & Amenities")
                    st.metric("Amenities Count", amenities_count, f"Max: 5")
                    st.metric("Property Age", f"{age_of_property} years", 
                             "New" if is_new_property == 1 else "Old" if is_old_property == 1 else "Mid")
                    st.metric("Readiness", "Ready" if is_ready == 1 else "Under Construction")
                    st.metric("Security", "Yes" if has_security_bin == 1 else "No")
                    
                    
                st.markdown("## 💡 Recommendations & Next Steps")
                
                if investment_class == 1:
                    st.info("""
                    **🎯 Strong Buy Recommendation:**
                    
                    1. **Immediate Actions:**
                       - Schedule property visit within 7 days
                       - Get professional valuation report
                       - Check builder/owner reputation
                    
                    2. **Negotiation Strategy:**
                       - Target price: ₹{}L - ₹{}L
                       - Highlight: {}, {}, {}
                    
                    3. **Long-term Strategy:**
                       - Hold for minimum 5 years
                       - Consider rental income potential
                       - Monitor infrastructure developments
                    
                    **📈 Expected Returns:**
                    - 5-year appreciation: {}%
                    - Annual growth: {}%
                    - Break-even period: ~{} years
                    """.format(
                        current_price*0.95, current_price*0.98,
                        "Good location" if infrastructure_score > 0.7 else "Average location",
                        "Premium amenities" if amenities_count >= 4 else "Basic amenities",
                        "Ready to move" if is_ready == 1 else "Future delivery",
                        appreciation_percent, annual_growth,
                        3 if annual_growth > 10 else 4 if annual_growth > 7 else 5
                    ))
                else:
                    st.info("""
                    **⚠️ Consider with Caution:**
                    
                    1. **Risk Assessment:**
                       - Price {}% {} market average
                       - Infrastructure score: {}/10
                       - Amenities: {}/5
                    
                    2. **Alternative Options:**
                       - Compare with 3-5 similar properties
                       - Explore nearby localities
                       - Consider different property types
                    
                    3. **If Proceeding:**
                       - Negotiate hard (target {}% discount)
                       - Get thorough inspection
                       - Verify all approvals and documents
                    
                    **📉 Risk Factors:**
                    - Lower than average growth potential
                    - {} infrastructure development
                    - {} rental demand
                    """.format(
                        abs(price_per_sqft - 0.09)*1000, "above" if price_per_sqft > 0.09 else "below",
                        int(infrastructure_score * 10), amenities_count,
                        10 if price_per_sqft > 0.1 else 5,
                        "Slow" if infrastructure_score < 0.5 else "Moderate",
                        "Lower" if amenities_count < 3 else "Moderate"
                    ))
                    
                #-------------------Download Reports----------------------------
                    
                st.markdown("## 📥 Export Analysis")
                
                report_content = f"""
                REAL ESTATE INVESTMENT ANALYSIS REPORT
                =======================================
                
                PROPERTY DETAILS:
                - Location: {city}, {state}
                - Property Type: {property_type}
                - BHK: {bhk}
                - Size: {size_sqft} sqft
                - Current Price: ₹{current_price:.1f}L
                - Year Built: {year_built}
                - Age: {age_of_property} years
                
                PREDICTION RESULTS:
                - Future Price (5 Years): ₹{future_price:.1f}L
                - Total Appreciation: {appreciation_percent:.1f}%
                - Annual Growth Rate: {annual_growth:.1f}%
                - Investment Rating: {'GOOD INVESTMENT' if investment_class == 1 else 'AVERAGE INVESTMENT'}
                - Confidence Score: {investment_prob*100:.1f}%
                
                KEY METRICS:
                - Price per SqFt: ₹{price_per_sqft:.4f}L
                - Infrastructure Score: {infrastructure_score:.2f}/1.0
                - Amenities Count: {amenities_count}/5
                - Transport Access: {transport_access}
                - School Density: {school_density:.1f}
                - Hospital Density: {hospital_density:.1f}
                
                RECOMMENDATION:
                {'STRONG BUY - Property shows excellent fundamentals and growth potential.' if investment_class == 1 else 'CAUTION ADVISED - Consider alternatives or negotiate better price.'}
                
                ANALYSIS DATE: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
                """
                
                st.download_button(
                    label="📄 Download Detailed Report (TXT)",
                    data=report_content,
                    file_name=f"property_analysis_{city}_{property_type}_{datetime.now().strftime('%Y%m%d')}.txt",
                    mime="text/plain"
                )
            except Exception as e:
                st.error(f"Error in prediction: {str(e)}")
                
    else:
            st.markdown("""
            ## 🏠 Welcome to Property Investment Analyzer

            This AI-powered tool analyzes real estate investments using machine learning models 
            trained on **250,000+ Indian properties**.

            ### 📋 How to Use:
            1. **Fill** property details in the sidebar
            2. **Click** "Analyze Property" 
            3. **Get** instant predictions and insights

            ### 🎯 What You'll Get:
            - ✅ **Future Price Prediction** (5 years)
            - ✅ **Investment Recommendation** (Good/Average)
            - ✅ **Annual Growth Rate** (CAGR)
            - ✅ **Visual Insights** & Charts
            - ✅ **Detailed Analysis** & Recommendations
            - ✅ **Market Comparisons**

            *All predictions are based on historical trends and should be used as guidance only.*
            """)
            if st.checkbox("Show Sample Analysis"):
                    sample_col1, sample_col2, sample_col3 = st.columns(3)
                    with sample_col1:
                        st.metric("Sample Property", "3 BHK Apartment", "Bangalore")
                    with sample_col2:
                        st.metric("Current Price", "₹275L", "")
                    with sample_col3:
                        st.metric("Predicted 5Y", "₹458L", "+66.5%")

# ==================== PAGE 2: MARKET INSIGHTS ====================
elif page == "📈 Market Insights":
    st.markdown('<h2 class="sub-header">📈 Market Insights & Trends</h2>', unsafe_allow_html=True)
    
    if df_clean is not None:
        # Filters
        st.sidebar.header("🔍 Filter Data")
        selected_state = st.sidebar.multiselect("Select State(s)", sorted(df_clean['State'].unique()), 
                                               default=['Maharashtra', 'Karnataka'])
        selected_city = st.sidebar.multiselect("Select City(s)", sorted(df_clean['City'].unique()),
                                              default=['Mumbai', 'Bangalore'])
        selected_type = st.sidebar.multiselect("Property Type", df_clean['Property_Type'].unique(),
                                              default=df_clean['Property_Type'].unique())
        
        # Filter data
        filtered_df = df_clean[
            (df_clean['State'].isin(selected_state)) &
            (df_clean['City'].isin(selected_city)) &
            (df_clean['Property_Type'].isin(selected_type))
        ]
        
        if len(filtered_df) > 0:
            # Market Overview Metrics
            st.markdown("### 📊 Market Overview")
            col1, col2, col3, col4 = st.columns(4)
            with col1:
                st.metric("Avg Price", f"₹{filtered_df['Price_in_Lakhs'].mean():.1f}L")
            with col2:
                st.metric("Avg Size", f"{filtered_df['Size_in_SqFt'].mean():.0f} sqft")
            with col3:
                st.metric("Avg Price/SqFt", f"₹{filtered_df['Price_per_SqFt'].mean():.3f}L")
            with col4:
                st.metric("Good Investment %", f"{(filtered_df['Good_Investment'].mean()*100):.1f}%")
            
            # Visualizations
            tab1, tab2, tab3, tab4 = st.tabs(["📈 Price Trends", "🏙️ City Analysis", "🏠 Property Types", "📊 Investment Analysis"])
            
            with tab1:
                col1, col2 = st.columns(2)
                with col1:
                    # Price by Year Built
                    year_price = filtered_df.groupby('Year_Built')['Price_in_Lakhs'].mean().reset_index()
                    fig = px.line(year_price, x='Year_Built', y='Price_in_Lakhs',
                                 title='Average Price by Construction Year',
                                 markers=True)
                    st.plotly_chart(fig, use_container_width=True)
                
                with col2:
                    # Price Distribution
                    fig = px.histogram(filtered_df, x='Price_in_Lakhs', nbins=50,
                                      title='Price Distribution',
                                      color_discrete_sequence=['#2196F3'])
                    st.plotly_chart(fig, use_container_width=True)
            
            with tab2:
                # City-wise Analysis
                city_stats = filtered_df.groupby('City').agg({
                    'Price_in_Lakhs': 'mean',
                    'Size_in_SqFt': 'mean',
                    'Good_Investment': 'mean',
                    'Price_per_SqFt': 'mean'
                }).round(2).reset_index()
                
                col1, col2 = st.columns(2)
                with col1:
                    fig = px.bar(city_stats.sort_values('Price_in_Lakhs', ascending=False).head(10),
                                x='City', y='Price_in_Lakhs',
                                title='Top 10 Cities by Average Price',
                                color='Price_in_Lakhs',
                                color_continuous_scale='Viridis')
                    st.plotly_chart(fig, use_container_width=True)
                
                with col2:
                    fig = px.scatter(city_stats, x='Price_in_Lakhs', y='Good_Investment',
                                    size='Size_in_SqFt', color='City',
                                    title='Price vs Investment Potential',
                                    hover_data=['City', 'Price_per_SqFt'])
                    st.plotly_chart(fig, use_container_width=True)
            
            with tab3:
                # Property Type Analysis
                type_stats = filtered_df.groupby('Property_Type').agg({
                    'Price_in_Lakhs': ['mean', 'median', 'count'],
                    'Good_Investment': 'mean'
                }).round(2)
                type_stats.columns = ['Avg Price', 'Median Price', 'Count', 'Good Investment %']
                type_stats['Good Investment %'] = type_stats['Good Investment %'] * 100
                
                col1, col2 = st.columns(2)
                with col1:
                    st.dataframe(type_stats.style.format({
                        'Avg Price': '₹{:.1f}L',
                        'Median Price': '₹{:.1f}L',
                        'Good Investment %': '{:.1f}%'
                    }).background_gradient(cmap='YlOrBr'), height=300)
                
                with col2:
                    fig = px.pie(filtered_df, names='Property_Type',
                                title='Property Type Distribution',
                                hole=0.4)
                    st.plotly_chart(fig, use_container_width=True)
            
            with tab4:
                # Investment Analysis
                col1, col2 = st.columns(2)
                with col1:
                    # Good vs Not Good distribution
                    fig = px.pie(filtered_df, names='Good_Investment',
                                title='Investment Quality Distribution',
                                labels={'0': 'Not Good', '1': 'Good'},
                                hole=0.4)
                    st.plotly_chart(fig, use_container_width=True)
                
                with col2:
                    # Factors affecting investment
                    factors_df = pd.DataFrame({
                        'Factor': ['Price/SqFt < 0.09', 'BHK ≥ 3', 'Infrastructure ≥ 0.7', 
                                  'Amenities ≥ 4', 'Ready to Move', 'Has Security'],
                        'Good Investment %': [
                            filtered_df[filtered_df['Price_per_SqFt'] < 0.09]['Good_Investment'].mean() * 100,
                            filtered_df[filtered_df['BHK'] >= 3]['Good_Investment'].mean() * 100,
                            filtered_df[filtered_df['Infrastructure_Score'] >= 0.7]['Good_Investment'].mean() * 100,
                            filtered_df[filtered_df['Amenities_Count'] >= 4]['Good_Investment'].mean() * 100,
                            filtered_df[filtered_df['Is_Ready'] == 1]['Good_Investment'].mean() * 100,
                            filtered_df[filtered_df['Has_Security'] == 1]['Good_Investment'].mean() * 100
                        ]
                    })
                    fig = px.bar(factors_df, x='Factor', y='Good Investment %',
                                title='Factors Affecting Investment Quality',
                                color='Good Investment %',
                                color_continuous_scale='Greens')
                    st.plotly_chart(fig, use_container_width=True)
        else:
            st.warning("No data available for selected filters.")
    else:
        st.warning("Cleaned data not available for market insights.")

# ==================== PAGE 3: DATA EXPLORER ====================
elif page == "🔍 Data Explorer":
    st.markdown('<h2 class="sub-header">🔍 Data Explorer</h2>', unsafe_allow_html=True)
    
    if df_clean is not None:
        # Data filters
        st.sidebar.header("🔍 Data Filters")
        
        # Numeric filters
        col1, col2 = st.sidebar.columns(2)
        with col1:
            min_price = st.number_input("Min Price (₹L)", 
                                       value=float(df_clean['Price_in_Lakhs'].min()),
                                       min_value=0.0)
            max_price = st.number_input("Max Price (₹L)",
                                       value=float(df_clean['Price_in_Lakhs'].max()),
                                       min_value=0.0)
        with col2:
            min_size = st.number_input("Min Size (SqFt)",
                                      value=float(df_clean['Size_in_SqFt'].min()),
                                      min_value=0.0)
            max_size = st.number_input("Max Size (SqFt)",
                                      value=float(df_clean['Size_in_SqFt'].max()),
                                      min_value=0.0)
        
        # Categorical filters
        selected_states = st.sidebar.multiselect("States", sorted(df_clean['State'].unique()))
        selected_cities = st.sidebar.multiselect("Cities", sorted(df_clean['City'].unique()))
        selected_bhk = st.sidebar.multiselect("BHK", sorted(df_clean['BHK'].unique()))
        
        # Apply filters
        filtered_data = df_clean.copy()
        filtered_data = filtered_data[
            (filtered_data['Price_in_Lakhs'] >= min_price) &
            (filtered_data['Price_in_Lakhs'] <= max_price) &
            (filtered_data['Size_in_SqFt'] >= min_size) &
            (filtered_data['Size_in_SqFt'] <= max_size)
        ]
        
        if selected_states:
            filtered_data = filtered_data[filtered_data['State'].isin(selected_states)]
        if selected_cities:
            filtered_data = filtered_data[filtered_data['City'].isin(selected_cities)]
        if selected_bhk:
            filtered_data = filtered_data[filtered_data['BHK'].isin(selected_bhk)]
        
        # Display data
        st.write(f"**Showing {len(filtered_data):,} properties** (Total: {len(df_clean):,})")
        
        # Data tabs
        tab1, tab2, tab3 = st.tabs(["📋 Raw Data", "📊 Statistics", "🔍 Search & Filter"])
        
        with tab1:
            # Show raw data with pagination
            page_size = 100
            page_number = st.number_input("Page", min_value=1, 
                                         max_value=len(filtered_data)//page_size + 1, 
                                         value=1)
            start_idx = (page_number - 1) * page_size
            end_idx = start_idx + page_size
            
            st.dataframe(filtered_data.iloc[start_idx:end_idx].style.format({
                'Price_in_Lakhs': '₹{:.1f}L',
                'Future_Price_5Y': '₹{:.1f}L',
                'Price_per_SqFt': '₹{:.4f}L',
                'Infrastructure_Score': '{:.2f}'
            }), height=400)
            
            # Download filtered data
            csv = filtered_data.to_csv(index=False)
            st.download_button(
                label="📥 Download Filtered Data (CSV)",
                data=csv,
                file_name=f"filtered_properties_{datetime.now().strftime('%Y%m%d')}.csv",
                mime="text/csv"
            )
        
        with tab2:
            # Statistics
            col1, col2 = st.columns(2)
            with col1:
                st.write("**Numerical Statistics:**")
                st.dataframe(filtered_data.select_dtypes(include=[np.number]).describe().round(2))
            
            with col2:
                st.write("**Categorical Statistics:**")
                for col in filtered_data.select_dtypes(include=['object']).columns:
                    if col in ['State', 'City', 'Property_Type', 'Furnished_Status']:
                        counts = filtered_data[col].value_counts().head(10)
                        st.write(f"**{col}:**")
                        st.write(counts)
        
        with tab3:
            # Advanced search
            st.write("### 🔍 Advanced Search")
            search_query = st.text_input("Search in all columns (comma-separated terms):")
            
            if search_query:
                search_terms = [term.strip().lower() for term in search_query.split(',')]
                mask = pd.Series(False, index=filtered_data.index)
                
                for term in search_terms:
                    if term:
                        for col in filtered_data.columns:
                            if filtered_data[col].dtype == 'object':
                                mask = mask | filtered_data[col].astype(str).str.lower().str.contains(term)
                
                search_results = filtered_data[mask]
                st.write(f"**Found {len(search_results)} properties matching your search**")
                st.dataframe(search_results.head(50))
    
    else:
        st.warning("Cleaned data not available for exploration.")

# ==================== PAGE 4: MODEL INFO ====================
elif page == "⚙️ Model Info":
    st.markdown('<h2 class="sub-header">⚙️ Model Information & Performance</h2>', unsafe_allow_html=True)
    
    col1, col2 = st.columns(2)
    
    with col1:
        st.markdown("### 🎯 Regression Model")
        st.markdown("""
        **Model:** Gradient Boosting Regressor  
        **Target:** Future_Price_5Y (5-year price prediction)  
        **Training Samples:** 200,000 properties  
        **Test Samples:** 50,000 properties  
        
        **Performance Metrics:**
        - **R² Score:** 0.9973 (99.73% variance explained)
        - **RMSE:** 11.89 Lakhs
        - **MAE:** 8.41 Lakhs
        - **Max Error:** ~50 Lakhs
        
        **Key Features:**
        1. Current Price (Most important)
        2. City/Location
        3. Infrastructure Score
        4. Amenities Count
        5. Property Type
        """)
        
        if reg_model is not None:
            fi_reg = get_feature_importance(reg_model, feature_columns, top_n=10)
            if fi_reg is not None:
                st.write("**Top 10 Important Features:**")
                st.dataframe(fi_reg.style.format({'Importance': '{:.4f}'}))
    
    with col2:
        st.markdown("### 🎯 Classification Model")
        st.markdown("""
        **Model:** XGBoost Classifier  
        **Target:** Good_Investment (Binary classification)  
        **Training Samples:** 200,000 properties  
        **Test Samples:** 50,000 properties  
        
        **Performance Metrics:**
        - **Accuracy:** 99.69%
        - **Precision:** 99.56%
        - **Recall:** 99.68%
        - **F1 Score:** 99.62%
        - **ROC AUC:** 1.000 (Perfect)
        
        **Classification Threshold:**
        - **Good Investment:** Probability ≥ 0.5
        - **Not Good:** Probability < 0.5
        
        **Key Decision Factors:**
        1. Price per SqFt
        2. Current Price
        3. Amenities Count
        4. Infrastructure Score
        5. BHK Configuration
        """)
        
        if clf_model is not None:
            fi_clf = get_feature_importance(clf_model, feature_columns, top_n=10)
            if fi_clf is not None:
                st.write("**Top 10 Important Features:**")
                st.dataframe(fi_clf.style.format({'Importance': '{:.4f}'}))
    
    st.markdown("---")
    st.markdown("### 📊 Model Comparison")
    
    # Model comparison metrics
    comparison_data = pd.DataFrame({
        'Model': ['Gradient Boosting', 'XGBoost', 'Random Forest', 'Linear Regression', 'Logistic Regression'],
        'Type': ['Regression', 'Classification', 'Both', 'Regression', 'Classification'],
        'Metric': ['R²: 0.9973', 'Accuracy: 99.69%', 'R²: 0.9969 / Acc: 99.54%', 'R²: 0.9852', 'Accuracy: 90.40%'],
        'Training Time': ['Medium', 'Fast', 'Slow', 'Very Fast', 'Very Fast'],
        'Interpretability': ['Medium', 'Medium', 'High', 'High', 'High']
    })
    
    st.dataframe(comparison_data.style.highlight_max(subset=['Metric'], color='lightgreen'))
    
    st.markdown("---")
    st.markdown("### 📁 Artifacts Information")
    
    artifacts_info = pd.DataFrame({
        'File': ['best_regression_model.pkl', 'best_classification_model.pkl', 'scaler.pkl', 
                'label_encoders.pkl', 'feature_columns.pkl', 'india_housing_prices_cleaned.csv'],
        'Size': ['~50 MB', '~45 MB', '~5 MB', '~2 MB', '~1 KB', '~150 MB'],
        'Description': ['Gradient Boosting model', 'XGBoost model', 'Feature scaler', 
                       'Categorical encoders', 'Feature column order', 'Cleaned dataset with targets']
    })
    
    st.dataframe(artifacts_info)

# Footer
st.markdown("---")
st.markdown(
    """
    <div style='text-align: center'>
    <p>🏠 <b>Real Estate Investment Advisor</b> | Machine Learning Project | Built with Streamlit & Scikit-learn</p>
    <p><small>Disclaimer: Predictions are for informational purposes only. Always consult with financial advisors before making investment decisions.</small></p>
    <p><small>© 2024 | Dataset: 250,000 Indian Properties | Models: Gradient Boosting & XGBoost</small></p>
    </div>
    """,
    unsafe_allow_html=True
)


Overwriting steamlit_real_estate.py


In [ ]:
!streamlit run steamlit_real_estate.py


  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://192.168.29.81:8501

2026-03-24 22:45:35.368 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-24 22:45:35.397 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-24 22:45:35.438 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-03-24 22:45:35.458 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

F